In [24]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

from langgraph.graph import  StateGraph, START, END
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [25]:
load_dotenv()

True

In [26]:
llm = ChatOllama(model="llama3.2")

In [27]:
import pypdf
loader = PyPDFLoader("intro-to-ml.pdf")
docs = loader.load()

In [28]:
len(docs)

392

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
len(chunks)

973

In [36]:
print("Total chunks:", len(chunks))

print(chunks[0].page_content)

Total chunks: 973
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y thon   
A GUIDE FOR DATA SCIENTISTS


In [38]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector = embeddings.embed_query(
    "What is Machine Learning?"
)

print(len(vector))
print(vector[:5])

768
[0.0057258, -0.008521418, -0.10270477, -0.028922176, 0.05779894]


In [39]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ml_documents",
    persist_directory="./chroma_db"
)

In [40]:
retriver = vector_store.as_retriever(search_type="similarity", search_kwargs={'k':4})

In [41]:
@tool 
def rag_tool(query):
    """
    Retriver relavent information from the pdf document.
    Use this tool when the user asks factual /conceptual qquestions that mighrt be answered from the stored documents.
    """
    result = retriver.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return{
        'query': query, 
        'context': context, 
        'metadata': metadata
    }


In [44]:
len(retriver.invoke("What is Machine Learing?"))


4

In [47]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [48]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [49]:
def chat_node(state: ChatState): 
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}